### Setup

In [1]:
# !pip install librecommender faker tensorflow torch

In [2]:
# !pip install tensorflow==2.15.0 keras==2.15.0 librecommender==1.5.2

In [3]:
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from collections import defaultdict
from faker import Faker
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from libreco.data import DatasetPure
from libreco.algorithms import UserCF, ItemCF, ALS, BPR

Instructions for updating:
non-resource variables are not supported in the long term


In [4]:
spark = (SparkSession.builder
    .appName("JobRecommender")
    .config("spark.sql.catalog.nessie.ref", "test")
    .getOrCreate()
)

In [5]:
jobs_df = spark.read.format("iceberg").load("nessie.silver.jobs")
jobs_df.show(5)

+--------------------+--------------+------------+--------------------+--------------+--------------------+----------+----------+--------+-----------+---------+---------+---------------+------------------+--------------+------------------+-------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------+
|             job_key|processed_date|    platform|         title_clean|location_clean|       company_clean|min_salary|max_salary|currency|salary_type|min_years|max_years|experience_type|education_standard|level_standard|work_form_standard|quantity_normalized|expired_date_norm|         tech_skills|         soft_skills|          skills_all| category_name_final|         description|         requirement|                link|gold_processed|
+--------------------+--------------+------------+--------------------+--------------+--------------------+----------+--

### Lấy dữ liệu việc làm mới nhất

In [6]:
w = Window.partitionBy(
    "job_key"
).orderBy(
    F.col("processed_date").desc()
)

latest_jobs = (
    jobs_df
    .withColumn(
        "rn",
        F.row_number().over(w)
    )
    .filter(
        F.col("rn") == 1
    )
    .drop("rn")
)

In [7]:
jobs_df = (
    latest_jobs
    .select(
        "job_key",
        "location_clean",
        "min_salary",
        "max_salary",
        "min_years",
        "max_years",
        "work_form_standard",
        "category_name_final",
        "skills_all"
    )
).toPandas()

jobs_df.head()

,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all
0,00036788d38de1c9d1433e9f36e8ba0de40233bf0ce39c...,Hà Nội,15000000.0,30000000.0,2.0,2.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[React Next JS, Node JS, React JS Java Script,..."
1,001aa442de3e83d685c55fad31227c883c098c0729395f...,Hà Nội,8000000.0,NaN,0.0,0.0,full_time,HOẠT ĐỘNG DỊCH VỤ KHÁC,"[Word, Excel, giao tiếp tiếng Trung, giao tiếp]"
2,002862efca2eef130d31db889706d06c4ae73a2e501642...,Đà Nẵng,50000000.0,NaN,0.0,0.0,full_time,HOẠT ĐỘNG KINH DOANH BẤT ĐỘNG SẢN,[Giao tiếp]
3,003c504f4f710bba48f2b05c5d98971057ddfd7745e04d...,Hồ Chí Minh,NaN,NaN,0.0,1.0,full_time,XÂY DỰNG,"[Auto CAD, triển khai bản vẽ shop drawing CAD ..."
4,003d99eae190f5178c817588b18b722adfbd9dc0aed4cc...,Hà Nội,15000000.0,30000000.0,0.0,0.0,full_time,GIÁO DỤC VÀ ĐÀO TẠO,"[giao tiếp, thuyết phục, chốt sale, Làm việc n..."


### Mapping skill


In [8]:
skill_map_df = pd.read_csv("notebooks/skill_freq_6k.csv", on_bad_lines='skip')

skill_lookup = {
    row["raw_skill_norm"].strip().lower(): row["canonical"].strip()
    for _, row in skill_map_df.iterrows()
    if pd.notna(row["canonical"])
}

In [9]:
import re

def normalize_skill(s: str) -> str | None:
    """Chuẩn hóa 1 skill string, trả về canonical hoặc None nếu không có trong dict."""
    s = str(s).strip().lower()
    s = re.sub(r'\s+', ' ', s)  # collapse whitespace
    return skill_lookup.get(s)  # None nếu không match

def normalize_skills_list(skills) -> list:
    """Normalize toàn bộ list skills của 1 job, bỏ skill không có trong dict."""
    if not hasattr(skills, '__iter__'):
        return []
    result = []
    seen = set()
    for s in skills:
        canonical = normalize_skill(s)
        if canonical and canonical not in seen:
            seen.add(canonical)
            result.append(canonical)
    return result

In [10]:
# Thay thế skills_all bằng version đã normalize
jobs_df["skills_all"] = jobs_df["skills_all"].apply(normalize_skills_list)

# Chỉ giữ job có ít nhất 1 skill sau normalize
jobs_df = jobs_df[jobs_df["skills_all"].apply(len) > 0].copy()

# Dùng skills_norm thay cho skills_all ở mọi chỗ
jobs_by_category = defaultdict(list)
for _, job in jobs_df.iterrows():
    jobs_by_category[job["category_name_final"]].append(job)

### Sinh dữ liệu tương tác người dùng

#### Gom job theo category

In [11]:
jobs_by_category = defaultdict(list)

for _, job in jobs_df.iterrows():
    jobs_by_category[
        job["category_name_final"]
    ].append(job)

#### Sinh user

In [12]:

# =========================
# 1. Clean jobs
# =========================

def clean_skill_list(skills):
    if not isinstance(skills, list):
        return []

    cleaned = []

    for s in skills:
        s = str(s).strip().lower()

        if not s:
            continue

        cleaned.append(s)

    return list(dict.fromkeys(cleaned))


jobs_df = jobs_df.copy()

jobs_df["skills_clean"] = jobs_df["skills_all"].apply(
    clean_skill_list
)

jobs_df = jobs_df[
    jobs_df["skills_clean"].apply(lambda x: len(x) > 0)
].copy()

jobs_df = jobs_df.dropna(
    subset=["category_name_final"]
)

# =========================
# 2. Build category pools
# =========================

jobs_by_category = defaultdict(list)

for _, job in jobs_df.iterrows():
    jobs_by_category[job["category_name_final"]].append(job)

MIN_JOBS_PER_CATEGORY = 300

valid_categories = [
    cate
    for cate, rows in jobs_by_category.items()
    if len(rows) >= MIN_JOBS_PER_CATEGORY
]

print("Valid categories:", len(valid_categories))

# =========================
# 3. Balanced user categories
# =========================

N_USERS = 10000

users_per_category = N_USERS // len(valid_categories)

user_categories = []

for cate in valid_categories:
    user_categories.extend(
        [cate] * users_per_category
    )

while len(user_categories) < N_USERS:
    user_categories.append(
        random.choice(valid_categories)
    )

random.shuffle(user_categories)

# =========================
# 4. Salary fallback stats
# =========================

category_salary_stats = (
    jobs_df
    .groupby("category_name_final")
    .agg({
        "min_salary": "median",
        "max_salary": "median"
    })
)

global_min_salary = jobs_df["min_salary"].median()
global_max_salary = jobs_df["max_salary"].median()

if pd.isna(global_min_salary):
    global_min_salary = 10_000_000

if pd.isna(global_max_salary):
    global_max_salary = 25_000_000

# =========================
# 5. Helper functions
# =========================

def safe_salary(category, col, default_value):
    value = np.nan

    if category in category_salary_stats.index:
        value = category_salary_stats.loc[
            category,
            col
        ]

    if pd.isna(value):
        value = default_value

    return value


def safe_exp(value, default_value):
    if pd.isna(value):
        return default_value

    return value


# =========================
# 6. Generate users
# =========================

users = []

for i in range(N_USERS):

    category = user_categories[i]
    category_jobs = jobs_by_category[category]

    seed_count = min(
        random.randint(2, 3),
        len(category_jobs)
    )

    seed_jobs = random.sample(
        category_jobs,
        seed_count
    )

    primary_job = random.choice(seed_jobs)

    # -----------------------
    # skill profile
    # -----------------------

    primary_skills = (
        primary_job["skills_clean"]
        if isinstance(primary_job["skills_clean"], list)
        else []
    )

    related_skill_pool = set(primary_skills)

    for job in seed_jobs:
        skills = job["skills_clean"]

        if isinstance(skills, list):
            related_skill_pool.update(skills)

    related_skill_pool = list(related_skill_pool)

    if len(primary_skills) > 0:

        min_base = max(
            1,
            int(len(primary_skills) * 0.6)
        )

        max_base = len(primary_skills)

        n_base = random.randint(
            min_base,
            max_base
        )

        base_skills = random.sample(
            primary_skills,
            n_base
        )

    else:
        base_skills = []

    extra_pool = list(
        set(related_skill_pool) - set(base_skills)
    )

    n_extra = random.randint(
        0,
        min(3, len(extra_pool))
    )

    extra_skills = random.sample(
        extra_pool,
        n_extra
    )

    user_skills = list(
        dict.fromkeys(base_skills + extra_skills)
    )

    # fallback nếu vẫn quá ít skill
    if len(user_skills) < 2 and len(related_skill_pool) > 0:
        user_skills = random.sample(
            related_skill_pool,
            min(3, len(related_skill_pool))
        )

    # -----------------------
    # experience profile
    # -----------------------

    min_exp = safe_exp(
        primary_job["min_years"],
        0
    )

    max_exp = safe_exp(
        primary_job["max_years"],
        min_exp + 2
    )

    lower = max(
        0,
        int(min_exp) - 1
    )

    upper = max(
        lower + 1,
        int(max_exp) + 1
    )

    years_exp = random.randint(
        lower,
        upper
    )

    # -----------------------
    # salary profile
    # -----------------------

    base_min_salary = safe_salary(
        category,
        "min_salary",
        global_min_salary
    )

    base_max_salary = safe_salary(
        category,
        "max_salary",
        global_max_salary
    )

    if base_max_salary < base_min_salary:
        base_max_salary = base_min_salary * 1.5

    desired_min_salary = int(
        base_min_salary *
        random.uniform(0.8, 1.0)
    )

    desired_max_salary = int(
        base_max_salary *
        random.uniform(1.0, 1.2)
    )

    if desired_max_salary <= desired_min_salary:
        desired_max_salary = int(
            desired_min_salary *
            random.uniform(1.2, 1.6)
        )

    # -----------------------
    # location
    # -----------------------

    locations = [
        job["location_clean"]
        for job in seed_jobs
        if pd.notna(job["location_clean"])
    ]

    location = (
        random.choice(locations)
        if locations
        else "Không xác định"
    )

    # -----------------------
    # work form
    # -----------------------

    work_forms = [
        job["work_form_standard"]
        for job in seed_jobs
        if pd.notna(job["work_form_standard"])
    ]

    preferred_work_form = (
        random.choice(work_forms)
        if work_forms
        else "unknown"
    )

    users.append({
        "user_id": f"U{i:05d}",
        "location": location,
        "preferred_category": category,
        "years_experience": years_exp,
        "desired_min_salary": desired_min_salary,
        "desired_max_salary": desired_max_salary,
        "preferred_work_form": preferred_work_form,
        "skills": user_skills,
        "seed_job_key": primary_job["job_key"]
    })

users_df = pd.DataFrame(users)

users_df.head()

Valid categories: 15


,user_id,location,preferred_category,years_experience,desired_min_salary,desired_max_salary,preferred_work_form,skills,seed_job_key
0,U00000,Hồ Chí Minh,"HOẠT ĐỘNG TÀI CHÍNH, NGÂN HÀNG VÀ BẢO HIỂM",4,12181176,23948793,full_time,"[làm việc nhóm, làm việc độc lập, luật thuế, c...",c402fc4a76e4adcbd8e1ba6dde9fe543f7796e1c84a599...
1,U00001,Hà Nội,BÁN BUÔN VÀ BÁN LẺ,6,11541224,22065132,full_time,"[autocad, xử lý tình huống, tiếng anh, quản lý...",330305104134bbfb4df089b17a0e830dc33fef502d678b...
2,U00002,Hà Nội,XÂY DỰNG,2,13590284,20573041,full_time,"[thiết kế hệ thống cấp thoát nước, mep, autoca...",4f303aa5743c2322f1bac49309f50b2039863cb8c33379...
3,U00003,Hồ Chí Minh,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ",2,11983413,20865900,full_time,"[__skip__, haccp, phát triển sản phẩm, microso...",fdcb869c64c474cc40195389359bec7ef4af63a0a63ba4...
4,U00004,Hồ Chí Minh,BÁN BUÔN VÀ BÁN LẺ,2,11638421,23925006,full_time,"[phân tích dữ liệu, __skip__, tiếng anh, tư du...",656f377a6a3d59587e15cc88da74a188903b5c4fc54391...


Sinh user sao cho rải đều các category

In [13]:
users_df["preferred_category"].value_counts()

preferred_category
BÁN BUÔN VÀ BÁN LẺ                                                                                        669
VẬN TẢI, KHO BÃI                                                                                          668
GIÁO DỤC VÀ ĐÀO TẠO                                                                                       668
HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ VẤN, CƠ SỞ HẠ TẦNG MÁY TÍNH VÀ CÁC DỊCH VỤ THÔNG TIN KHÁC    667
HOẠT ĐỘNG DỊCH VỤ KHÁC                                                                                    667
Y TẾ VÀ HOẠT ĐỘNG TRỢ GIÚP XÃ HỘI                                                                         667
XÂY DỰNG                                                                                                  666
CÔNG NGHIỆP CHẾ BIẾN, CHẾ TẠO                                                                             666
HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ                                                          

In [14]:
users_spark = spark.createDataFrame(
    users_df
)

users_spark.write.format("iceberg").mode("overwrite").saveAsTable("nessie.silver.users")

In [15]:
users_spark.show(5)

+-------+-----------+--------------------+----------------+------------------+------------------+-------------------+--------------------+--------------------+
|user_id|   location|  preferred_category|years_experience|desired_min_salary|desired_max_salary|preferred_work_form|              skills|        seed_job_key|
+-------+-----------+--------------------+----------------+------------------+------------------+-------------------+--------------------+--------------------+
| U00000|Hồ Chí Minh|HOẠT ĐỘNG TÀI CHÍ...|               4|          12181176|          23948793|          full_time|[làm việc nhóm, l...|c402fc4a76e4adcbd...|
| U00001|     Hà Nội|  BÁN BUÔN VÀ BÁN LẺ|               6|          11541224|          22065132|          full_time|[autocad, xử lý t...|330305104134bbfb4...|
| U00002|     Hà Nội|            XÂY DỰNG|               2|          13590284|          20573041|          full_time|[thiết kế hệ thốn...|4f303aa5743c2322f...|
| U00003|Hồ Chí Minh|HOẠT ĐỘNG CHUYÊN ..

#### Sinh interaction

In [16]:
def skill_match_ratio(
    user_skills,
    job_skills
):

    if not user_skills or not job_skills:
        return 0

    user_skills = set(user_skills)
    job_skills = set(job_skills)

    overlap = len(
        user_skills &
        job_skills
    )

    return overlap / len(job_skills)

In [17]:
def calculate_match_score(
    user,
    job
):

    score = 0

    # 0 -> 10
    skill_ratio = skill_match_ratio(
        user["skills"],
        job["skills_all"]
    )

    score += skill_ratio * 10

    # same category
    if (
        user["preferred_category"]
        ==
        job["category_name_final"]
    ):
        score += 3

    # work form
    if (
        user["preferred_work_form"]
        ==
        job["work_form_standard"]
    ):
        score += 1

    # location
    if (
        user["location"]
        ==
        job["location_clean"]
    ):
        score += 1

    # experience
    min_years = (
        job["min_years"]
        if pd.notna(job["min_years"])
        else 0
    )

    max_years = (
        job["max_years"]
        if pd.notna(job["max_years"])
        else min_years + 2
    )

    if (
        min_years
        <=
        user["years_experience"]
        <=
        max_years
    ):
        score += 2

    return score

In [ ]:
interactions = []

job_records = jobs_df.to_dict("records")

jobs_by_category = defaultdict(list)

for job in job_records:
    jobs_by_category[job["category_name_final"]].append(job)

rating_map = {
    "view": 1,
    "save": 3,
    "apply": 5
}

for idx, user in users_df.iterrows():

    if idx % 500 == 0:
        print(f"Processing user {idx}/{len(users_df)}")

    user_category = user["preferred_category"]

    same_category_jobs = jobs_by_category.get(
        user_category,
        []
    )

    sampled_jobs = []

    # 90% cùng ngành
    sampled_jobs.extend(
        random.sample(
            same_category_jobs,
            min(45, len(same_category_jobs))
        )
    )

    # 10% ngoài ngành nhưng có liên quan skill
    random_jobs = random.sample(
        job_records,
        min(30, len(job_records))
    )

    related_other_jobs = []

    for job in random_jobs:

        if job["category_name_final"] == user_category:
            continue

        skill_ratio = skill_match_ratio(
            user["skills"],
            job["skills_all"]
        )

        if skill_ratio > 0:
            related_other_jobs.append(job)

    sampled_jobs.extend(
        random.sample(
            related_other_jobs,
            min(5, len(related_other_jobs))
        )
    )

    # remove duplicate
    sampled_jobs = list({
        job["job_key"]: job
        for job in sampled_jobs
    }.values())

    for job in sampled_jobs:

        score = calculate_match_score(
            user,
            job
        )

        if score < 5:
            continue

        # Nếu khác ngành, yêu cầu skill overlap mạnh hơn
        if job["category_name_final"] != user_category:

            ratio = skill_match_ratio(
                user["skills"],
                job["skills_all"]
            )

            if ratio < 0.3:
                continue

        probability = min(
            score / 16,
            0.9
        )

        if random.random() > probability:
            continue

        if score >= 12:
            event = random.choices(
                ["view", "save", "apply"],
                weights=[15, 30, 55]
            )[0]

        elif score >= 9:
            event = random.choices(
                ["view", "save", "apply"],
                weights=[45, 45, 10]
            )[0]

        else:
            event = random.choices(
                ["view", "save"],
                weights=[85, 15]
            )[0]

        interactions.append({
            "user_id": user["user_id"],
            "job_key": job["job_key"],
            "event_type": event,
            "rating": rating_map[event],
            "event_time": datetime.now() - timedelta(
                days=random.randint(0, 180),
                hours=random.randint(0, 23),
                minutes=random.randint(0, 59)
            )
        })

interactions_df = pd.DataFrame(interactions)

print(interactions_df.shape)
interactions_df["event_type"].value_counts(normalize=True)

Processing user 0/10000
Processing user 500/10000
Processing user 1000/10000
Processing user 1500/10000
Processing user 2000/10000
Processing user 2500/10000
Processing user 3000/10000
Processing user 3500/10000
Processing user 4000/10000
Processing user 4500/10000
Processing user 5000/10000
Processing user 5500/10000
Processing user 6000/10000
Processing user 6500/10000
Processing user 7000/10000
Processing user 7500/10000
Processing user 8000/10000
Processing user 8500/10000
Processing user 9000/10000
Processing user 9500/10000
(73090, 5)


event_type
view    0.848858
save    0.151142
Name: proportion, dtype: float64

In [19]:
interaction_spark = spark.createDataFrame(
    interactions_df
)

interaction_spark.write.format("iceberg").mode("overwrite").saveAsTable("nessie.silver.interactions")

In [20]:
interaction_spark.show(5)

+-------+--------------------+----------+------+--------------------+
|user_id|             job_key|event_type|rating|          event_time|
+-------+--------------------+----------+------+--------------------+
| U00000|3d1e6adfdd8de5b8d...|      save|     3|2026-04-09 08:51:...|
| U00000|51a611b92aa3f99bf...|      view|     1|2026-03-20 05:27:...|
| U00000|dd0109999fe3744fa...|      view|     1|2026-01-14 04:28:...|
| U00000|c18c31395e8c4cca3...|      save|     3|2026-03-01 09:32:...|
| U00000|c1ec82eaedcdbe53f...|      view|     1|2026-05-15 19:59:...|
+-------+--------------------+----------+------+--------------------+
only showing top 5 rows



### (Optional) Lấy dữ liệu từ bảng nếu đã sinh dữ liệu

In [ ]:
users_df = spark.read.format("iceberg").load("nessie.silver.users").toPandas()
interactions_df = spark.read.format("iceberg").load("nessie.silver.interactions").toPandas()

### (Optional) Lấy dữ liệu từ ở ngoài

In [ ]:
# Load users and interactions from csv files (if not using Iceberg)
users_df = pd.read_csv("notebooks/users.csv")
interactions_df = pd.read_csv("notebooks/interactions.csv")

In [ ]:
import json

users_df["skills"] = users_df["skills"].apply(
    json.loads
)

### Tiền xử lý interactions

In [21]:
interaction_cf = (
    interactions_df
    .groupby(["user_id", "job_key"])["rating"]
    .max()
    .reset_index()
)

interaction_cf = interaction_cf.rename(
    columns={
        "user_id": "user",
        "job_key": "item",
        "rating": "label"
    }
)

interaction_cf.head()

,user,item,label
0,U00000,3d1e6adfdd8de5b8d8eb6d629c9fbc11798e5343c54eb5...,3
1,U00000,51a611b92aa3f99bf978966d99503632db1d12588053a6...,1
2,U00000,7004e3247cfd8683c9c8b4497f69b029eeebcfa270c715...,3
3,U00000,c18c31395e8c4cca3287a4726d7011da7e7490015b5f1a...,3
4,U00000,c1ec82eaedcdbe53f41777cf4bae2ade003aee8c7b2e4a...,1


### Tạo train/test

In [22]:
from sklearn.model_selection import train_test_split

train_parts = []
test_parts = []

for user, group in interaction_cf.groupby("user"):

    if len(group) < 2:
        train_parts.append(group)
        continue

    train_g, test_g = train_test_split(
        group,
        test_size=0.2,
        random_state=42
    )

    train_parts.append(train_g)
    test_parts.append(test_g)

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

test_df = test_df[
    test_df["item"].isin(train_df["item"])
].copy()

In [23]:
train_data, data_info = DatasetPure.build_trainset(
    train_df
)

test_data = DatasetPure.build_testset(
    test_df,
    data_info
)

#### Tạo tập train / test riêng cho BPR

In [24]:
ranking_df = interaction_cf.copy()
ranking_df["label"] = 1

train_rank_df, test_rank_df = train_test_split(
    ranking_df,
    test_size=0.2,
    random_state=42
)

train_rank_data, rank_data_info = DatasetPure.build_trainset(
    train_rank_df
)

test_rank_data = DatasetPure.build_testset(
    test_rank_df,
    rank_data_info
)

### Train mô hình

#### UserCF

In [25]:
usercf = UserCF(
    task="rating",
    data_info=data_info,
    sim_type="cosine",
    k_sim=100
)

usercf.fit(
    train_data,
    neg_sampling=False,
    verbose=2
)

Training start time: 2026-06-11 07:45:07
Final block size and num: (9836, 1)
sim_matrix elapsed: 0.337s
sim_matrix, shape: (9836, 9836), num_elements: 138552, density: 0.1432 %


top_k: 100%|██████████| 9836/9836 [00:00<00:00, 141601.99it/s]

#### ItemCF

In [26]:
itemcf = ItemCF(
    task="rating",
    data_info=data_info,
    sim_type="cosine",
    k_sim=100
)

itemcf.fit(
    train_data,
    neg_sampling=False,
    verbose=2
)

Training start time: 2026-06-11 07:45:08
Final block size and num: (6023, 5)
sim_matrix elapsed: 2.277s
sim_matrix, shape: (30113, 30113), num_elements: 322984, density: 0.0356 %


top_k: 100%|██████████| 30113/30113 [00:00<00:00, 199521.79it/s]

#### ALS

In [27]:
als = ALS(
    task="rating",
    data_info=data_info,
    embed_size=32,
    n_epochs=20,
    reg=1e-4,
    seed=42
)

als.fit(
    train_data,
    neg_sampling=False,
    verbose=2
)

Training start time: 2026-06-11 07:45:10
Epoch 1 elapsed: 0.085s
Epoch 2 elapsed: 0.066s
Epoch 3 elapsed: 0.067s
Epoch 4 elapsed: 0.062s
Epoch 5 elapsed: 0.060s
Epoch 6 elapsed: 0.060s
Epoch 7 elapsed: 0.063s
Epoch 8 elapsed: 0.065s
Epoch 9 elapsed: 0.068s
Epoch 10 elapsed: 0.069s
Epoch 11 elapsed: 0.070s
Epoch 12 elapsed: 0.061s
Epoch 13 elapsed: 0.057s
Epoch 14 elapsed: 0.063s
Epoch 15 elapsed: 0.064s
Epoch 16 elapsed: 0.060s
Epoch 17 elapsed: 0.060s
Epoch 18 elapsed: 0.059s
Epoch 19 elapsed: 0.061s
Epoch 20 elapsed: 0.059s


#### BPR

In [28]:
bpr = BPR(
    task="ranking",
    data_info=rank_data_info,
    embed_size=32,
    n_epochs=20,
    lr=0.001,
    reg=1e-4,
    seed=42
)

bpr.fit(
    train_rank_data,
    neg_sampling=True,
    verbose=2
)

Training start time: 2026-06-11 07:45:12


train: 100%|██████████| 229/229 [00:01<00:00, 225.92it/s]


Epoch 1 elapsed: 1.017s
	 train_loss: 0.6927


train: 100%|██████████| 229/229 [00:00<00:00, 245.16it/s]


Epoch 2 elapsed: 0.937s
	 train_loss: 0.6894


...

Epoch 18 elapsed: 1.043s
	 train_loss: 0.6681


train: 100%|██████████| 229/229 [00:01<00:00, 213.28it/s]


Epoch 19 elapsed: 1.076s
	 train_loss: 0.6681


train: 100%|██████████| 229/229 [00:00<00:00, 235.28it/s]


Epoch 20 elapsed: 0.976s
	 train_loss: 0.6678


### Recommend cho 1 user

Chọn user U00123

In [29]:
user_id = "U00003"
user_idx = 3

In [30]:
users_df.loc[user_idx]

user_id                                                           U00003
location                                                     Hồ Chí Minh
preferred_category           HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ
years_experience                                                       2
desired_min_salary                                              11983413
desired_max_salary                                              20865900
preferred_work_form                                            full_time
skills                 [__skip__, haccp, phát triển sản phẩm, microso...
seed_job_key           fdcb869c64c474cc40195389359bec7ef4af63a0a63ba4...
Name: 3, dtype: object

In [31]:
for skill in users_df.loc[user_idx, "skills"]:
    print(skill)

__skip__
haccp
phát triển sản phẩm
microsoft office
phân tích
xử lý tình huống
quản lý dự án
kiểm toán


#### Thử nghiệm trên 4 mô hình

In [32]:
user_recs = usercf.recommend_user(
    user=user_id,
    n_rec=20
)

user_res = user_recs[user_id]
enriched_user = jobs_df[
    jobs_df["job_key"].isin(user_res)
]
enriched_user['method'] = 'user'
enriched_user.head()

/tmp/ipykernel_40718/850884489.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_user['method'] = 'user'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,skills_clean,method
4002,38e3fe561ad4a2dc964a82b4d9c7394a00afe1b4968bba...,Quảng Ninh,NaN,NaN,1.0,1.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ","[__SKIP__, Hệ thống điện]","[__skip__, hệ thống điện]",user
9080,c2efd264bc23bb2d8984dc46de5d21f86941c1c4d75016...,Hồ Chí Minh,13000000.0,16000000.0,1.0,2.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ","[Tiếng Anh, Microsoft Word, Excel]","[tiếng anh, microsoft word, excel]",user
13946,38ccf4c44892a98680fbb78b77c2834640cf97f47896d5...,Bình Dương,15000000.0,25000000.0,1.0,5.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ","[AutoCAD, Giao tiếp, Làm việc nhóm]","[autocad, giao tiếp, làm việc nhóm]",user
14388,59eeb7a244608e0e4ea2f425936f38ba115368b5f7cb31...,Hồ Chí Minh,NaN,NaN,1.0,3.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ","[Dự toán, AutoCAD, Excel, Đọc hiểu tài liệu kỹ...","[dự toán, autocad, excel, đọc hiểu tài liệu kỹ...",user
18214,84b9286ec8006fae0320aa45c60943e50169c217727ad2...,Hồ Chí Minh,NaN,NaN,5.0,5.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ","[Tư duy phân tích, Governance, Quản lý rủi ro,...","[tư duy phân tích, governance, quản lý rủi ro,...",user


In [33]:
item_recs = itemcf.recommend_user( 
    user=user_id,
    n_rec=20
)

item_res = item_recs[user_id]
enriched_item = jobs_df[
    jobs_df["job_key"].isin(item_res)
]
enriched_item['method'] = 'item'
enriched_item.head()

/tmp/ipykernel_40718/3104140752.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_item['method'] = 'item'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,skills_clean,method
11335,70da6b1f7f7e0c66a23e3f5bda3eb38c5aa22fb49d8d0b...,Bình Dương,10000000.0,12000000.0,0.0,0.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ","[Quản lý chất lượng, ISO 9001, IATF, Microsoft...","[quản lý chất lượng, iso 9001, iatf, microsoft...",item
13346,08b0f487363a255a8ea95f9883c71d1184b3d7a3e88c3e...,Đồng Nai,NaN,NaN,2.0,2.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ","[Vận hành máy CNC, Tiếng Anh]","[vận hành máy cnc, tiếng anh]",item
18214,84b9286ec8006fae0320aa45c60943e50169c217727ad2...,Hồ Chí Minh,NaN,NaN,5.0,5.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ","[Tư duy phân tích, Governance, Quản lý rủi ro,...","[tư duy phân tích, governance, quản lý rủi ro,...",item
20269,23cd22dec07086655e30a4ddbf0727c4f49e18bc288263...,Bình Dương,15000000.0,20000000.0,1.0,2.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ",[Tiếng Trung],[tiếng trung],item
21482,831298792a4a91bf9f4033ed64fa4d800881d1b382debd...,Hồ Chí Minh,NaN,NaN,2.0,3.0,full_time,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ","[Xây dựng, Đọc bản vẽ kỹ thuật, Hệ thống điện,...","[xây dựng, đọc bản vẽ kỹ thuật, hệ thống điện,...",item


In [34]:
als_recs = als.recommend_user(
    user=user_id,   
    n_rec=20    
)

als_res = als_recs[user_id]
enriched_als = jobs_df[
    jobs_df["job_key"].isin(als_res)    
]
enriched_als['method'] = 'als'
enriched_als.head()

/tmp/ipykernel_40718/1723471370.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_als['method'] = 'als'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,skills_clean,method
554,2d1a09a7e7ab359c9822fbdf61a070c55917cea2b79399...,Hồ Chí Minh,8000000.0,12000000.0,0.0,1.0,full_time,HOẠT ĐỘNG DỊCH VỤ KHÁC,"[Quản lý kho, Làm việc nhóm]","[quản lý kho, làm việc nhóm]",als
3019,eca3d59c4dd5ed7f2ec5f2856d62a99df795c11f4691ef...,Hà Nội,NaN,NaN,0.0,1.0,full_time,HOẠT ĐỘNG KINH DOANH BẤT ĐỘNG SẢN,"[TikTok, Google, Facebook, Zalo, Làm việc nhóm...","[tiktok, google, facebook, zalo, làm việc nhóm...",als
7488,476d802dd7ad833da8996a445203ff790a4119630c2f7f...,Hà Nội,20000000.0,50000000.0,3.0,3.0,full_time,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...","[Business Analysis, Phân tích dữ liệu, Tiếng Anh]","[business analysis, phân tích dữ liệu, tiếng anh]",als
12093,a9f87e6a290b3f3a95b99616b7194ede726005ec8eede3...,Hồ Chí Minh,12000000.0,NaN,2.0,2.0,full_time,"NGHỆ THUẬT, THỂ THAO VÀ GIẢI TRÍ","[Canva, Photoshop, Adobe Premiere, __SKIP__, V...","[canva, photoshop, adobe premiere, __skip__, v...",als
15871,cd727cb39e56252ffd08758aa1bb5bc6d138c457204be6...,Hồ Chí Minh,NaN,NaN,0.0,0.0,full_time,"CÔNG NGHIỆP CHẾ BIẾN, CHẾ TẠO",[Industrial Engineering],[industrial engineering],als


In [35]:
bpr_recs = bpr.recommend_user(
    user=user_id,
    n_rec=20
)

bpr_res = bpr_recs[user_id]
enriched_bpr = jobs_df[
    jobs_df["job_key"].isin(bpr_res)
]
enriched_bpr['method'] = 'bpr'
enriched_bpr.head()

/tmp/ipykernel_40718/2677664316.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  enriched_bpr['method'] = 'bpr'


,job_key,location_clean,min_salary,max_salary,min_years,max_years,work_form_standard,category_name_final,skills_all,skills_clean,method
1286,658f0813f22f7261369b3a6180e72e2ce87f1075714da7...,Hồ Chí Minh,NaN,NaN,2.0,5.0,full_time,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,"[Tiếng Việt, Tiếng Anh, Excel, Microsoft Word,...","[tiếng việt, tiếng anh, excel, microsoft word,...",bpr
1420,710db2aa0310b35f05f8453945bcc52a48b6bfe2d2df2d...,Hồ Chí Minh,15000000.0,20000000.0,1.0,3.0,full_time,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,"[Thu ngân, Xử lý từ chối, Làm việc nhóm, Giao ...","[thu ngân, xử lý từ chối, làm việc nhóm, giao ...",bpr
3882,312c34c8eb5231a4b9e4553b9d8d4210371f5c2aa93d86...,Hà Nội,NaN,NaN,1.0,2.0,full_time,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,"[Tổng hợp, Tiếng Anh, Tiếng Đức, Tiếng Pháp, E...","[tổng hợp, tiếng anh, tiếng đức, tiếng pháp, e...",bpr
4390,58d0877e245c1f0876ba8827c4018548460cab245c094a...,Hồ Chí Minh,NaN,NaN,2.0,5.0,full_time,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,"[Tiếng Trung, hàn điện, Tiếng Nhật, Tiếng Anh,...","[tiếng trung, hàn điện, tiếng nhật, tiếng anh,...",bpr
5026,8861df9ce4aa333d690738dff0f354319a35028531e898...,Hà Nội,10000000.0,20000000.0,1.0,2.0,full_time,DỊCH VỤ LƯU TRÚ VÀ ĂN UỐNG,"[__SKIP__, Marketing, Bán hàng, Giao tiếp, Lập...","[__skip__, marketing, bán hàng, giao tiếp, lập...",bpr


### Đánh giá mô hình

#### BPR

In [36]:
from libreco.evaluation import evaluate

evaluate(
    model=bpr,
    data=test_rank_data,
    metrics=[
        "precision",
        "recall",
        "ndcg"
    ],
    neg_sampling=True,
    k=10
)

eval_listwise: 100%|██████████| 7305/7305 [02:08<00:00, 56.85it/s] 


{'precision': 0.0006433949349760438,
 'recall': 0.0034519735341090574,
 'ndcg': 0.003138612630469002}

#### UserCF

In [37]:
evaluate(
    model=usercf,
    data=test_data,
    neg_sampling=False,
    metrics=[
        "rmse",
        "mae"
    ]
)

eval_pointwise:   0%|          | 0/2 [00:00<?, ?it/s]

No common interaction or similar neighbor for user 9516 and item 8608, proceed with default prediction
No common interaction or similar neighbor for user 5694 and item 3831, proceed with default prediction
No common interaction or similar neighbor for user 3935 and item 2459, proceed with default prediction
No common interaction or similar neighbor for user 5184 and item 878, proceed with default prediction
No common interaction or similar neighbor for user 1178 and item 3152, proceed with default prediction
No common interaction or similar neighbor for user 2222 and item 106, proceed with default prediction


eval_pointwise: 100%|██████████| 2/2 [00:00<00:00,  5.82it/s]


{'rmse': 0.7603603942351793, 'mae': 0.510861783367504}

#### ItemCF

In [38]:
evaluate(
    model=itemcf,
    data=test_data,
    neg_sampling=False,
    metrics=[
        "rmse",
        "mae"
    ]
)

eval_pointwise:   0%|          | 0/2 [00:00<?, ?it/s]

No common interaction or similar neighbor for user 9516 and item 8608, proceed with default prediction
No common interaction or similar neighbor for user 5694 and item 3831, proceed with default prediction
No common interaction or similar neighbor for user 3935 and item 2459, proceed with default prediction
No common interaction or similar neighbor for user 5184 and item 878, proceed with default prediction
No common interaction or similar neighbor for user 1178 and item 3152, proceed with default prediction
No common interaction or similar neighbor for user 2222 and item 106, proceed with default prediction


eval_pointwise: 100%|██████████| 2/2 [00:00<00:00,  5.65it/s]


{'rmse': 0.7620063132394884, 'mae': 0.5116490836953859}

#### ALS

In [39]:
evaluate(
    model=als,
    data=test_data,
    neg_sampling=False,
    metrics=[
        "rmse",
        "mae"
    ]
)

eval_pointwise: 100%|██████████| 2/2 [00:00<00:00, 263.68it/s]


{'rmse': 0.7745558, 'mae': 0.30826828}

### Đánh giá chất lượng đề xuất

#### Avg Skill Overlap

In [55]:
def avg_skill_overlap(
    recommended_jobs,
    user_skills
):

    overlaps = []

    for skills in recommended_jobs["skills_clean"]:

        overlap = len(
            set(user_skills)
            &
            set(skills)
        )

        overlaps.append(overlap)

    return np.mean(overlaps)

In [56]:
print ("UserCF skill overlap:",
    avg_skill_overlap(
        enriched_user,
        users_df.loc[user_idx, "skills"]
    )
)

print ("ItemCF skill overlap:",
    avg_skill_overlap(
        enriched_item,
        users_df.loc[user_idx, "skills"]
    )
)

print ("ALS skill overlap:",
    avg_skill_overlap(
        enriched_als,
        users_df.loc[user_idx, "skills"]
    )
)

print ("BPR skill overlap:",
    avg_skill_overlap(
        enriched_bpr,
        users_df.loc[user_idx, "skills"]
    )
)

UserCF skill overlap: 0.8
ItemCF skill overlap: 0.7
ALS skill overlap: 0.4
BPR skill overlap: 0.85


#### Avg Category Match

In [42]:
def avg_category_match(
    recommended_jobs,
    user_category
):

    return (
        recommended_jobs[
            "category_name_final"
        ]
        ==
        user_category
    ).mean()

In [43]:
print ("UserCF category m:",
    avg_category_match(
        enriched_user,
        users_df.loc[user_idx, "preferred_category"]
    )
)

print ("ItemCF category m:",
    avg_category_match(
        enriched_item,
        users_df.loc[user_idx, "preferred_category"]
    )
)

print ("ALS category m:",
    avg_category_match(
        enriched_als,
        users_df.loc[user_idx, "preferred_category"]
    )
)

print ("BPR category m:",
    avg_category_match(
        enriched_bpr,
        users_df.loc[user_idx, "preferred_category"]
    )
)

UserCF category m: 1.0
ItemCF category m: 1.0
ALS category m: 0.05
BPR category m: 0.0


#### Avg Experience Match

In [44]:
def avg_exp_match(
    recommended_jobs,
    years_exp
):

    matches = []

    for _, row in recommended_jobs.iterrows():

        min_exp = (
            row["min_years"]
            if pd.notna(row["min_years"])
            else 0
        )

        max_exp = (
            row["max_years"]
            if pd.notna(row["max_years"])
            else 100
        )

        matches.append(
            min_exp <= years_exp <= max_exp
        )

    return np.mean(matches)

In [45]:
print("UserCF exp match:",
    avg_exp_match(
        enriched_user,
        users_df.loc[user_idx, "years_experience"]
    )
)

print("ItemCF exp match:",
    avg_exp_match(
        enriched_item,
        users_df.loc[user_idx, "years_experience"]
    )
)   

print("ALS exp match:",
    avg_exp_match(
        enriched_als,
        users_df.loc[user_idx, "years_experience"]
    )
)

print("BPR exp match:",
    avg_exp_match(
        enriched_bpr,
        users_df.loc[user_idx, "years_experience"]
    )
)

UserCF exp match: 0.65
ItemCF exp match: 0.55
ALS exp match: 0.25
BPR exp match: 0.9


### (Optional) Checking

In [46]:
check_df = (
    interactions_df
    .merge(
        users_df[["user_id", "preferred_category", "skills"]],
        on="user_id",
        how="left"
    )
    .merge(
        jobs_df[["job_key", "category_name_final", "skills_all"]],
        on="job_key",
        how="left"
    )
)

In [47]:
check_df["same_category"] = (
    check_df["preferred_category"]
    ==
    check_df["category_name_final"]
)

check_df["same_category"].mean()

1.0

In [48]:
def skill_overlap_count(user_skills, job_skills):
    if not isinstance(user_skills, list) or not isinstance(job_skills, list):
        return 0

    user_set = set(
        str(s).strip().lower()
        for s in user_skills
    )

    job_set = set(
        str(s).strip().lower()
        for s in job_skills
    )

    return len(user_set & job_set)

In [49]:
check_df["skill_overlap"] = check_df.apply(
    lambda row: skill_overlap_count(
        row["skills"],
        row["skills_all"]
    ),
    axis=1
)

In [50]:
(check_df["skill_overlap"] > 0).mean()

0.5915036256669859

In [51]:
check_df.groupby("event_type")["skill_overlap"].mean()

event_type
save    0.931656
view    0.948745
Name: skill_overlap, dtype: float64

In [52]:
bad_cases = check_df[
    (check_df["same_category"] == False)
    &
    (check_df["skill_overlap"] == 0)
]

bad_cases[
    [
        "user_id",
        "preferred_category",
        "category_name_final",
        "skills",
        "skills_all",
        "event_type",
        "rating"
    ]
].head(20)

,user_id,preferred_category,category_name_final,skills,skills_all,event_type,rating


In [53]:
summary = pd.DataFrame({
    "same_category_rate": [check_df["same_category"].mean()],
    "has_skill_overlap_rate": [(check_df["skill_overlap"] > 0).mean()],
    "avg_skill_overlap": [check_df["skill_overlap"].mean()]
})

summary

,same_category_rate,has_skill_overlap_rate,avg_skill_overlap
0,1.0,0.591504,0.946162


In [54]:
users_df[
    users_df["skills"].apply(
        lambda skills:
            any(
                "python" in str(s).lower()
                for s in skills
            )
            if isinstance(skills, list)
            else False
    )
]

,user_id,location,preferred_category,years_experience,desired_min_salary,desired_max_salary,preferred_work_form,skills,seed_job_key
37,U00037,Hà Nội,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",7,11209024,21514639,full_time,"[tài chính, huấn luyện, thuyết phục, data anal...",f7c5664c883a10e2de03ab48525f7e66902d441f4a86bf...
60,U00060,Hà Nội,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",6,11665276,23362109,full_time,"[giao tiếp, windows, github, git, gitlab, java...",ab7a58e743ebe9ddacdca87f17619254dcb42f70e69ea6...
134,U00134,Hà Nội,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",0,12208728,23295633,full_time,"[unreal engine, python, autodesk maya, làm việ...",d443a496c0caa1d590dde1d41972961e02f99baf7f00b4...
178,U00178,Hải Dương,"HOẠT ĐỘNG CHUYÊN MÔN, KHOA HỌC VÀ CÔNG NGHỆ",6,11016620,18924127,full_time,"[scikit-learn, python, hugging face transforme...",48229533dea75858c23180b6847facc81a200ac7371d4b...
182,U00182,Hà Nội,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",3,12067726,20145336,full_time,"[tiếng anh, quản lý, giao tiếp, __skip__, lãnh...",2f72690677ffda052c71bdb310895507fa5e84a255f32a...
...,...,...,...,...,...,...,...,...,...
9734,U09734,Hà Nội,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",2,13556414,20591534,full_time,"[jenkins, javascript, java, trình bày, python,...",348ce3aec8da0544806dd243ffa79cac7e23f645b5294d...
9794,U09794,Khác,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",2,11311316,23137957,full_time,"[python, pytorch, tensorflow, erp, tổng hợp, p...",7cd43c9ff01740b694a8c46d41262b4f043cb39d5ef236...
9805,U09805,Hà Nội,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",2,12858071,22010401,full_time,"[system design, machine learning, ai, python]",65b442f7145ef9740f18636a4f15c20790ea3c7cd85a3c...
9894,U09894,Hà Nội,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",10,13769788,23798542,full_time,"[__skip__, làm việc độc lập và theo nhóm, phân...",908c92a2e260e98faf04bc0a7b50974c21934998e15fe7...


In [57]:
def precision_recall_at_k(model, test_df, train_df, k=10):
    results = []
    
    for user in test_df["user"].unique():
        # Ground truth: items user tương tác trong test
        ground_truth = set(test_df[test_df["user"] == user]["item"])
        
        # Items đã thấy trong train (loại khỏi recommend)
        seen = set(train_df[train_df["user"] == user]["item"])
        
        # Get top-K recommendations
        try:
            recs = model.recommend_user(user=user, n_rec=k)
            rec_items = set(recs[user])
        except:
            continue
        
        hits = len(ground_truth & rec_items)
        precision = hits / k
        recall = hits / len(ground_truth) if ground_truth else 0
        
        results.append({"precision": precision, "recall": recall})
    
    df = pd.DataFrame(results)
    return df.mean()

In [58]:
print("UserCF Precision@10, Recall@10:",
    precision_recall_at_k(
        usercf,
        test_df,
        train_df,
        k=10
    )
)

print("ItemCF Precision@10, Recall@10:",
    precision_recall_at_k(
        itemcf,
        test_df,
        train_df,
        k=10
    )
)

print("ALS Precision@10, Recall@10:",
    precision_recall_at_k(
        als,
        test_df,
        train_df,
        k=10
    )
)

UserCF Precision@10, Recall@10: precision    0.002378
recall       0.015125
dtype: float64
ItemCF Precision@10, Recall@10: precision    0.002391
recall       0.015324
dtype: float64
ALS Precision@10, Recall@10: precision    0.000119
recall       0.000837
dtype: float64


In [80]:
# found user that has any of these skills: Python, SQL, Docker, Python, Bash, AWS, ReactJS, Jenkins, Ubuntu. Match as many as possible.
def find_users_by_skills(users_df, target_skills):
    target_set = set(s.lower() for s in target_skills)
    
    def skill_match_count(skills):
        if not isinstance(skills, list):
            return 0
        skill_set = set(str(s).lower() for s in skills)
        return len(target_set & skill_set)
    
    users_df["match_count"] = users_df["skills"].apply(skill_match_count)
    matched_users = users_df[users_df["match_count"] > 0].copy()
    matched_users = matched_users.sort_values(by="match_count", ascending=False)
    
    return matched_users[["user_id", "skills", "match_count"]]

target_skills = ["Python", "SQL", "Docker", "Bash", "AWS", "ReactJS", "Jenkins", "Ubuntu"]
matched_users = find_users_by_skills(users_df, target_skills)   
matched_users.head(20)


,user_id,skills,match_count
8867,U08867,"[saas, script, python, ci/cd, a/b testing, bas...",5
9436,U09436,"[làm việc nhóm, giải quyết vấn đề, aws, bash, ...",4
1458,U01458,"[cloud computing, bash, network, kafka, docker...",4
1158,U01158,"[microsoft project, a/b testing, load balancin...",4
5879,U05879,"[node.js, restful api, angular, tiếng nhật, aw...",3
727,U00727,"[làm việc nhóm, phân tích dữ liệu, javascript,...",3
364,U00364,"[maven, tiếng anh, điều phối công việc, phân t...",3
6433,U06433,"[network, kubernetes, google cloud, __skip__, ...",3
573,U00573,"[__skip__, ci/cd, argo cd, monitoring, bash, r...",3
4244,U04244,"[scrum, jira, agile, làm việc nhóm, jmeter, au...",3


In [85]:
users_df[users_df["user_id"] == 'U08867']

,user_id,location,preferred_category,years_experience,desired_min_salary,desired_max_salary,preferred_work_form,skills,seed_job_key,match_count
8867,U08867,Hà Nội,"HOẠT ĐỘNG VIỄN THÔNG; LẬP TRÌNH MÁY TÍNH, TƯ V...",1,13310784,20981249,full_time,"[saas, script, python, ci/cd, a/b testing, bas...",fefa9423e65376d5d274b14692cedc113bcf821a451dfa...,5


In [86]:
# Get all skills of user U08867
user_skills = users_df.loc[
    users_df["user_id"] == 'U08867',    
    "skills"
].values[0]

print("Skills of user U08867:", user_skills)    

Skills of user U08867: ['saas', 'script', 'python', 'ci/cd', 'a/b testing', 'bash', 'load balancing', '__skip__', 'jenkins', 'microsoft project', 'optimization', 'tiếng anh', 'network', 'docker', 'aws', 'security', 'kubernetes', 'lập trình', 'microsoft azure', 'argo cd', 'automation', 'tableau', 'làm việc nhóm']
